In [ ]:
from datetime import datetime
import pandas as pd
from matplotlib import pyplot as plt
import pycountry
from os import listdir as ls
import numpy as np
from IPython.display import display, Markdown
import pycountry_convert as pc
from matplotlib_inline.backend_inline import set_matplotlib_formats

from emu_renewal.constants import OUTPUTS_PATH, DATA_PATH, MOB_COLOURS
from emu_renewal.plotting import MOB_SOURCE_MAP, compare_proc_mob, MOB_ANALYSIS_MAP, MOB_NAME_MAP, CONT_CMAP, get_proc_mob_corr, plot_proc_mob_corr
from emu_renewal.utils import get_countries_by_continent, get_mob_avail_countries
from emu_renewal.inputs import get_google_mobility, get_world_shp

set_matplotlib_formats("svg")

In [ ]:
job_path = OUTPUTS_PATH / "46090067"
all_countries = ls(job_path)
countries_by_cont = get_countries_by_continent(all_countries)
notes = {
    "g_mob": "Mobility values presented as Google data divided by 100, plus one.",
    "fb_visited_mob": "Mobility values presented as one plus Facebook data.",
    "fb_singletile_mob": "Mobility values presented as one minus Facebook data.",
}

# Purpose
This document presents the results of the variable process (with uncertainty)
implemented without scaling for mobility for each country.
It is intended to provide a raw comparison between the 
residual non-mechanistic variable process that needed
to be applied for each analysis in the absence of mobility scaling
and the various mobility data fields available from both Google and Facebook.

# Results by continent and country

In [ ]:
# import pycountry
# corrs_series = pd.Series(corrs)
# corrs_series.index = corrs_series.index.map(lambda c: pycountry.countries.lookup(c).name)
# corrs_series = corrs_series.sort_index()
# display(Markdown(corrs_series.to_latex()))

In [ ]:
for mob_type in MOB_ANALYSIS_MAP:
    mob_source = MOB_ANALYSIS_MAP[mob_type]
    mob_name = MOB_NAME_MAP[mob_type]
    mob_type_text = mob_type.replace("_", " ")
    section_title = f"## {mob_name} mobility comparison\n\n"
    display(Markdown(section_title))
    display(Markdown(notes[mob_source]))
    for cont in CONT_CMAP:
        countries = get_mob_avail_countries(countries_by_cont[cont], mob_source)
        display(Markdown(f"### {pc.convert_continent_code_to_continent_name(cont)}"))
        n_cols = 3 if cont in ["NA", "SA"] else 4
        if cont == "EU":
            display(compare_proc_mob(job_path, countries[:16], n_cols, mob_type))
            display(compare_proc_mob(job_path, countries[16:], n_cols, mob_type))
        else:
            display(compare_proc_mob(job_path, countries, n_cols, mob_type))
    corrs = {}
    for c in all_countries:
        try:
            corrs[c] = get_proc_mob_corr(job_path, c, mob_type)
        except:
            continue
    display(Markdown(f"### Correlations between the variable process (no mobility analysis) and {mob_type_text} mobility by country"))
    display(plot_proc_mob_corr(corrs))

# 